# California APR Explorer

Run All loads and validates the archived HCD APR 2018–2024 release. It never prepares source data or fits models. Model publication is a separate, owner-only manual release workflow.

Local dry-run (fixture release for notebook development):

```bash
python3 scripts/export_pages_catalog.py --fixture --staging-dir docs/data/releases/2018-2024
python3 scripts/verify_pages_catalog.py docs/data/releases/2018-2024
```


In [ ]:
import json
from pathlib import Path

repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'docs' / 'index.html').exists()), None)
if repo is None:
    raise FileNotFoundError('Run from the APR Explorer repository')
release = repo / 'docs' / 'data' / 'releases' / '2018-2024'
required = ['manifest.json', 'chart_labels.json', 'catalog.json', 'map_metrics.json', 'maps.geojson']
missing = [name for name in required if not (release / name).exists()]
if missing:
    raise FileNotFoundError(
        f'Archived release incomplete: {missing}. Run the fixture dry-run in the intro cell.'
    )
artifact_keys = {
    'manifest.json': 'manifest',
    'chart_labels.json': 'chart_labels',
    'catalog.json': 'catalog',
    'map_metrics.json': 'map_metrics',
    'maps.geojson': 'maps',
}
artifacts = {artifact_keys[name]: json.loads((release / name).read_text()) for name in required}
if artifacts['manifest'].get('release_id') != '2018-2024':
    raise ValueError('Unexpected release id')
if not artifacts['catalog'] or not artifacts['map_metrics'] or not artifacts['maps'].get('features'):
    raise ValueError('Malformed archived release')
print(
    f"Release {artifacts['manifest']['release_id']} built {artifacts['manifest'].get('built_at', 'unknown')}"
)


In [ ]:
pairs = [
    dict(zip(('geography', 'y_col', 'x_col', 'robustness'), key.split(':')), payload=payload)
    for key, payload in artifacts['catalog'].items()
]
missing_predictors = sorted({p['x_col'] for p in pairs} - set(artifacts['chart_labels']['predictors']))
missing_outcomes = sorted({p['y_col'] for p in pairs} - set(artifacts['chart_labels']['outcomes']))
print({'missing_predictors': missing_predictors, 'missing_outcomes': missing_outcomes})


## Maps


In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

geo_views = {
    'Incorporated cities': {'city'},
    'Whole counties': {'county_whole'},
    'Cities + unincorporated county': {'city', 'county_residual'},
}
map_subheaders = {
    'Incorporated cities': 'Incorporated city jurisdictions',
    'Whole counties': 'Whole county jurisdictions',
    'Cities + unincorporated county': 'Incorporated cities and unincorporated county jurisdictions',
}
geography_view = widgets.Dropdown(
    description='Geography view',
    options=list(geo_views),
    value='Cities + unincorporated county',
)
metric_by_title = {m['title']: m for m in artifacts['map_metrics']}
map_metric = widgets.Dropdown(description='Map metric', options=list(metric_by_title))
map_subheader = widgets.HTML()
map_output = widgets.Output()


def draw_map(*_):
    metric = metric_by_title[map_metric.value]
    features = [
        f
        for f in artifacts['maps']['features']
        if f['properties']['geo_type'] in geo_views[geography_view.value]
    ]
    geojson = {'type': 'FeatureCollection', 'features': features}
    div = metric['cmap_kind'] == 'div'
    trace = go.Choroplethmapbox(
        geojson=geojson,
        locations=[f['properties']['feature_id'] for f in features],
        z=[f['properties'].get(metric['metric_col']) for f in features],
        featureidkey='properties.feature_id',
        colorscale='RdYlGn' if div else 'PuRd',
        zmid=0 if div else None,
        marker={'opacity': 0.78},
        text=[f['properties'].get('city_name') or f['properties'].get('county_name') for f in features],
        hovertemplate='%{text}<br>%{z}<extra></extra>',
    )
    map_subheader.value = f"<p><em>{map_subheaders[geography_view.value]}</em></p>"
    with map_output:
        map_output.clear_output(wait=True)
        go.Figure(trace).update_layout(
            mapbox_style='carto-positron',
            mapbox_zoom=4.5,
            mapbox_center={'lat': 37.2, 'lon': -119.5},
            title=metric['title'],
        ).show()


geography_view.observe(draw_map, names='value')
map_metric.observe(draw_map, names='value')
display(widgets.HBox([geography_view, map_metric]), map_subheader, map_output)
draw_map()


## Models


In [ ]:
import pandas as pd

GEO_NAMES = {'city': 'City', 'zip': 'ZIP code'}
chart_labels = artifacts['chart_labels']
model_display = widgets.Dropdown(description='Model display')
zero_values = widgets.Dropdown(
    description='Zero Values',
    options=['Two-Part Hurdle', 'Positive Only'],
    value='Two-Part Hurdle',
)
selectors = {
    name: widgets.Dropdown(description=label)
    for name, label in [
        ('geography', 'Geography'),
        ('y_col', 'Outcome'),
        ('x_col', 'Predictor'),
        ('robustness', 'Robustness'),
    ]
}
model_output = widgets.Output()


def selector_label(name, value):
    if name == 'geography':
        return GEO_NAMES.get(value, value)
    if name == 'y_col':
        return chart_labels['outcomes'].get(value, value)
    if name == 'x_col':
        return chart_labels['predictors'].get(value, value)
    return value


def set_selector_options(name, values):
    current = selectors[name].value
    ordered = sorted(values)
    selectors[name].options = [(selector_label(name, value), value) for value in ordered]
    selectors[name].value = current if current in ordered else ordered[0]


def settle():
    pool = pairs
    for name in selectors:
        set_selector_options(name, {p[name] for p in pool})
        pool = [p for p in pool if p[name] == selectors[name].value]
    pair = pool[0]
    choices = ['Two-Part MLE + Stationary Bootstrap'] + (
        ['Hierarchical Bayes', 'Both'] if pair['payload']['availability']['hierarchical'] else []
    )
    old = model_display.value
    model_display.options = choices
    model_display.value = old if old in choices else choices[0]
    return pair['payload']


def add_component(fig, pair, component, name, color):
    x = pair['x_grid']
    fig.add_scatter(
        x=x, y=component['lower'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip'
    )
    fig.add_scatter(
        x=x,
        y=component['upper'],
        mode='lines',
        line={'width': 0},
        fill='tonexty',
        fillcolor=color.replace('1)', '0.18)'),
        name=f'{name} 95% interval',
        hoverinfo='skip',
    )
    fig.add_scatter(x=x, y=component['mean'], mode='lines', line={'color': color}, name=name)


def diagnostics_table(pair):
    stats = pair['stats']['two_part']
    return pd.DataFrame(
        [
            {'Part': 'Zero (logit)', 'Parameter': 'α', 'Estimate': stats['alpha'], 't': None, 'p': None},
            {
                'Part': 'Zero (logit)',
                'Parameter': 'β',
                'Estimate': stats['beta'],
                't': stats.get('beta_t'),
                'p': stats.get('beta_p'),
            },
            {'Part': 'Positive', 'Parameter': 'γ', 'Estimate': stats['intercept'], 't': None, 'p': None},
            {
                'Part': 'Positive',
                'Parameter': 'δ',
                'Estimate': stats['slope'],
                't': stats.get('slope_t'),
                'p': stats.get('slope_p'),
            },
        ]
    )


def draw_model(*_):
    pair = settle()
    view_key = 'two_part_hurdle' if zero_values.value == 'Two-Part Hurdle' else 'positive_only'
    view = pair['views'][view_key]
    obs = pair['observations']
    keep_zeros = zero_values.value == 'Two-Part Hurdle'
    keep = [i for i, y in enumerate(obs['y']) if keep_zeros or y > 0]
    fig = go.Figure(
        go.Scatter(
            x=[obs['x'][i] for i in keep],
            y=[obs['y'][i] for i in keep],
            text=[obs['labels'][i] for i in keep],
            mode='markers',
            name='Observations',
            marker={'color': ['#777' if obs['y'][i] == 0 else '#222' for i in keep]},
            hovertemplate='%{text}<br>x=%{x}<br>y=%{y}<extra></extra>',
        )
    )
    selected = model_display.value
    if selected != 'Hierarchical Bayes':
        add_component(fig, pair, view['stationary_bootstrap'], 'Stationary bootstrap', 'rgba(0,151,167,1)')
    if selected != 'Two-Part MLE + Stationary Bootstrap':
        add_component(fig, pair, view['hierarchical'], 'Hierarchical Bayes', 'rgba(194,24,91,1)')
    x_title = chart_labels['predictors'].get(pair['x_col'], pair['x_col'])
    y_title = f"{chart_labels['outcomes'].get(pair['y_col'], pair['y_col'])} {chart_labels['yRateSuffix']}"
    fig.update_layout(xaxis_title=x_title, yaxis_title=y_title)
    with model_output:
        model_output.clear_output(wait=True)
        display(diagnostics_table(pair))
        fig.show()


for control in [*selectors.values(), model_display, zero_values]:
    control.observe(draw_model, names='value')
display(
    widgets.VBox(
        [
            widgets.HBox(list(selectors.values())),
            widgets.HBox([model_display, zero_values]),
        ]
    ),
    model_output,
)
draw_model()
